# Bài 1: NumPy — mảng, broadcasting, vectorization

NumPy là **nền móng** của toàn bộ hệ sinh thái AI trong Python: Pandas, scikit-learn, PyTorch, TensorFlow đều xây trên (hoặc mô phỏng) `ndarray` của NumPy. Ảnh là array 3 chiều, batch dữ liệu là array 2 chiều, trọng số mạng neural là array — hiểu NumPy là hiểu "ngôn ngữ chung" của dữ liệu số.

Mục tiêu bài học:
- Hiểu vì sao NumPy nhanh hơn list thuần Python
- Tạo array và đọc `shape`, `dtype`, `ndim`
- Indexing, slicing, lọc bằng boolean mask
- Phép toán vectorized và hàm tổng hợp theo `axis`
- **Broadcasting** — quy tắc quan trọng nhất của NumPy
- `reshape`, transpose, ghép/chia array, sinh số ngẫu nhiên có `seed`

> Nhắc lại cách học: đọc lý thuyết → chạy thử ví dụ → tự làm bài tập, đừng xem lời giải trước.

## 1. Vì sao cần NumPy?

List của Python rất linh hoạt (chứa được mọi kiểu) nhưng chậm với tính toán số: mỗi phần tử là một object riêng, vòng `for` chạy bằng trình thông dịch. NumPy lưu số **liền kề trong bộ nhớ, cùng một kiểu**, và tính toán bằng code C đã biên dịch — nhanh hơn hàng chục đến hàng trăm lần.

In [1]:
import time
import numpy as np

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

# Bình phương toàn bộ phần tử: list comprehension (bài 7) vs NumPy
start = time.perf_counter()
squares_list = [x * x for x in py_list]
t_list = time.perf_counter() - start

start = time.perf_counter()
squares_np = np_arr * np_arr
t_np = time.perf_counter() - start

print(f"list : {t_list * 1000:.1f} ms")
print(f"numpy: {t_np * 1000:.1f} ms")
print(f"NumPy nhanh gấp ~{t_list / t_np:.0f} lần")

list : 30.2 ms
numpy: 2.4 ms
NumPy nhanh gấp ~13 lần


## 2. Tạo array và các thuộc tính cơ bản

Ba thuộc tính cần thuộc lòng:
- `shape`: kích thước từng chiều, dạng tuple, ví dụ `(3, 4)` = 3 hàng, 4 cột
- `ndim`: số chiều
- `dtype`: kiểu dữ liệu của phần tử (`int64`, `float64`, `bool`, ...)

In [2]:
a = np.array([1, 2, 3, 4])                    # từ list 1 chiều
m = np.array([[1, 2, 3], [4, 5, 6]])          # từ list lồng nhau -> 2 chiều

print(a, "| shape:", a.shape, "| ndim:", a.ndim, "| dtype:", a.dtype)
print(m)
print("shape:", m.shape, "| ndim:", m.ndim, "| size:", m.size)

# Mọi phần tử cùng một kiểu: có số thực là cả array thành float
print(np.array([1, 2, 3.5]).dtype)

# Đổi kiểu bằng astype (trả về array mới)
print(a.astype(np.float32))

[1 2 3 4] | shape: (4,) | ndim: 1 | dtype: int64
[[1 2 3]
 [4 5 6]]
shape: (2, 3) | ndim: 2 | size: 6
float64
[1. 2. 3. 4.]


### Các hàm tạo array thường dùng

In [3]:
print(np.zeros((2, 3)))            # toàn 0 -- truyền vào shape (tuple)
print(np.ones((2, 2)))             # toàn 1
print(np.full((2, 3), 7))          # toàn giá trị 7
print(np.eye(3))                   # ma trận đơn vị 3x3

print(np.arange(0, 10, 2))         # như range nhưng ra array: 0,2,4,6,8
print(np.linspace(0, 1, 5))        # 5 điểm chia đều từ 0 đến 1 (gồm cả 2 đầu)

[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1.]
 [1. 1.]]
[[7 7 7]
 [7 7 7]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


## 3. Indexing và slicing

Giống list (bài 4) nhưng mở rộng cho nhiều chiều: dùng `a[hàng, cột]` trong **một cặp ngoặc**, thay vì `a[hàng][cột]`.

⚠️ **Slice của NumPy là một "view"** — nó trỏ vào cùng vùng nhớ với array gốc, sửa slice sẽ sửa luôn array gốc. Muốn bản độc lập, dùng `.copy()`.

In [4]:
m = np.array([[10, 11, 12, 13],
              [20, 21, 22, 23],
              [30, 31, 32, 33]])

print(m[0, 1])        # hàng 0, cột 1 -> 11
print(m[1])           # cả hàng 1
print(m[:, 2])        # cả cột 2 (: nghĩa là "tất cả")
print(m[0:2, 1:3])    # khối con: hàng 0-1, cột 1-2
print(m[-1, -1])      # phần tử cuối cùng

# Slice là view -> sửa slice làm đổi array gốc
sub = m[0:2, 0:2]
sub[0, 0] = 999
print(m[0, 0])        # 999 !

# Muốn bản sao độc lập:
safe = m[0:2, 0:2].copy()
safe[0, 0] = -1
print(m[0, 0])        # vẫn 999

11
[20 21 22 23]
[12 22 32]
[[11 12]
 [21 22]]
33
999
999


### Boolean mask — lọc theo điều kiện

So sánh một array với giá trị cho ra array `True/False` cùng shape; dùng nó làm index để **chỉ giữ phần tử `True`**. Đây là phiên bản "list comprehension có `if`" của NumPy, nhưng nhanh hơn nhiều. Kết hợp nhiều điều kiện bằng `&` (và), `|` (hoặc), `~` (phủ định) — **bắt buộc có ngoặc** quanh từng điều kiện.

In [5]:
scores = np.array([0.45, 0.91, 0.72, 0.30, 0.88])

mask = scores >= 0.7
print(mask)               # [False  True  True False  True]
print(scores[mask])       # [0.91 0.72 0.88]

print(scores[(scores > 0.4) & (scores < 0.9)])

# Gán theo điều kiện: đặt mọi score < 0.5 về 0
clipped = scores.copy()
clipped[clipped < 0.5] = 0
print(clipped)

# Đếm / kiểm tra
print((scores >= 0.5).sum())    # True = 1 nên sum() chính là đếm
print(np.where(scores >= 0.5, 1, 0))   # như "1 if s >= 0.5 else 0" ở bài 7

[False  True  True False  True]
[0.91 0.72 0.88]
[0.45 0.72 0.88]
[0.   0.91 0.72 0.   0.88]
3
[0 1 1 0 1]


## 4. Vectorization: tính toán trên cả array, không cần vòng `for`

Phép toán `+ - * / **` và các hàm toán học (`np.sqrt`, `np.exp`, `np.log`, ...) áp lên **từng phần tử** một cách tự động. Quy tắc vàng khi viết NumPy: *nếu bạn đang viết vòng `for` qua phần tử array, gần như chắc chắn có cách tốt hơn.*

In [6]:
x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([10.0, 20.0, 30.0, 40.0])

print(x + y)          # cộng từng cặp phần tử
print(x * y)          # NHÂN TỪNG PHẦN TỬ (không phải nhân ma trận!)
print(x ** 2)
print(np.sqrt(x))
print(np.exp(x))

# Ví dụ AI: hàm sigmoid áp lên cả vector logits trong một dòng
logits = np.array([-2.0, 0.0, 3.0])
sigmoid = 1 / (1 + np.exp(-logits))
print(sigmoid)

[11. 22. 33. 44.]
[ 10.  40.  90. 160.]
[ 1.  4.  9. 16.]
[1.         1.41421356 1.73205081 2.        ]
[ 2.71828183  7.3890561  20.08553692 54.59815003]
[0.11920292 0.5        0.95257413]


### Hàm tổng hợp và tham số `axis`

`sum`, `mean`, `std`, `min`, `max`, `argmax`, ... mặc định gộp **toàn bộ** array thành 1 số. Truyền `axis` để gộp theo một chiều:
- `axis=0`: gộp **dọc theo hàng** → kết quả có 1 giá trị cho **mỗi cột**
- `axis=1`: gộp **dọc theo cột** → kết quả có 1 giá trị cho **mỗi hàng**

Mẹo nhớ: chiều nào được chỉ định thì chiều đó **biến mất** khỏi shape kết quả.

In [7]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])     # shape (2, 3)

print(m.sum())           # 21
print(m.sum(axis=0))     # [5 7 9]   -> shape (3,): tổng mỗi cột
print(m.sum(axis=1))     # [ 6 15]   -> shape (2,): tổng mỗi hàng
print(m.mean(axis=0))

# argmax: VỊ TRÍ của giá trị lớn nhất -- dùng để lấy nhãn dự đoán từ xác suất
probs = np.array([[0.1, 0.7, 0.2],
                  [0.8, 0.1, 0.1]])
print(probs.argmax(axis=1))    # [1 0] -> lớp dự đoán của từng mẫu

21
[5 7 9]
[ 6 15]
[2.5 3.5 4.5]
[1 0]


## 5. Broadcasting

Khi hai array **khác shape**, NumPy tự "kéo giãn" array nhỏ hơn cho khớp — **không copy thật** nên rất tiết kiệm.

Quy tắc: so sánh shape **từ chiều cuối về đầu**. Hai chiều tương thích nếu **bằng nhau** hoặc **một trong hai bằng 1** (hoặc không tồn tại). Nếu không, báo lỗi.

```
(3, 4)  +  (4,)    -> OK   (4,) được kéo thành (3, 4)
(3, 4)  +  (3, 1)  -> OK   cột (3,1) được kéo ngang thành (3, 4)
(3, 4)  +  (3,)    -> LỖI  4 != 3
```

In [8]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])         # (2, 3)

row = np.array([10, 20, 30])      # (3,)
print(m + row)                    # cộng vào MỖI HÀNG

col = np.array([[100], [200]])    # (2, 1)
print(m + col)                    # cộng vào MỖI CỘT

print(m * 2)                      # scalar cũng là broadcasting

# Lỗi khi shape không tương thích
try:
    m + np.array([1, 2])          # (2,3) + (2,) : 3 != 2
except ValueError as e:
    print("Lỗi:", e)

[[11 22 33]
 [14 25 36]]
[[101 102 103]
 [204 205 206]]
[[ 2  4  6]
 [ 8 10 12]]
Lỗi: operands could not be broadcast together with shapes (2,3) (2,) 


### Ứng dụng kinh điển: chuẩn hoá dữ liệu (standardization)

Với ma trận dữ liệu `X` shape `(n_mẫu, n_đặc_trưng)`, ta trừ trung bình và chia độ lệch chuẩn **của từng cột**. `X.mean(axis=0)` có shape `(n_đặc_trưng,)` và broadcasting làm phần còn lại. Đây chính là việc `StandardScaler` của scikit-learn (Module 5) làm bên trong.

In [9]:
X = np.array([[170.0, 65.0],
              [160.0, 50.0],
              [180.0, 80.0],
              [175.0, 70.0]])     # cột 0: chiều cao (cm), cột 1: cân nặng (kg)

mean = X.mean(axis=0)             # shape (2,)
std = X.std(axis=0)               # shape (2,)
X_norm = (X - mean) / std         # (4,2) - (2,) -> broadcasting

print("mean:", mean)
print("std :", std)
print(X_norm.round(2))
print("mean sau chuẩn hoá:", X_norm.mean(axis=0).round(6))   # ~0
print("std  sau chuẩn hoá:", X_norm.std(axis=0))             # 1

mean: [171.25  66.25]
std : [ 7.39509973 10.82531755]
[[-0.17 -0.12]
 [-1.52 -1.5 ]
 [ 1.18  1.27]
 [ 0.51  0.35]]
mean sau chuẩn hoá: [ 0. -0.]
std  sau chuẩn hoá: [1. 1.]


## 6. Thay đổi hình dạng: reshape, transpose, `newaxis`

- `reshape(...)`: đổi shape, **tổng số phần tử phải giữ nguyên**. Dùng `-1` để NumPy tự suy ra một chiều.
- `.T`: chuyển vị (đổi hàng ↔ cột).
- `np.newaxis` (hoặc `None`): thêm một chiều kích thước 1 — hay dùng để chuẩn bị broadcasting.
- `flatten()`: duỗi thành 1 chiều.

In [10]:
a = np.arange(12)                  # [0 1 2 ... 11], shape (12,)

print(a.reshape(3, 4))
print(a.reshape(2, 6).shape)
print(a.reshape(-1, 3).shape)      # -1: tự tính -> (4, 3)

m = a.reshape(3, 4)
print(m.T.shape)                   # (4, 3)
print(m.flatten().shape)           # (12,)

v = np.array([1, 2, 3])            # (3,)
print(v[:, np.newaxis].shape)      # (3, 1)  -- vector cột
print(v[np.newaxis, :].shape)      # (1, 3)  -- vector hàng

# Ví dụ: ảnh 28x28 -> vector 784 chiều để đưa vào mô hình
image = np.zeros((28, 28))
print(image.reshape(-1).shape)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
(4, 3)
(4, 3)
(12,)
(3, 1)
(1, 3)
(784,)


### Ghép và chia array

In [11]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print(np.concatenate([a, b], axis=0))   # nối theo chiều hàng -> (4, 2)
print(np.concatenate([a, b], axis=1))   # nối theo chiều cột  -> (2, 4)
print(np.stack([a, b]).shape)           # xếp chồng thành chiều MỚI -> (2, 2, 2)

parts = np.split(np.arange(6), 3)       # chia thành 3 phần bằng nhau
print(parts)

[[1 2]
 [3 4]
 [5 6]
 [7 8]]
[[1 2 5 6]
 [3 4 7 8]]
(2, 2, 2)
[array([0, 1]), array([2, 3]), array([4, 5])]


## 7. Số ngẫu nhiên và `seed`

Trong AI ta cần số ngẫu nhiên liên tục (khởi tạo trọng số, xáo trộn dữ liệu, chia train/test). Dùng `np.random.default_rng(seed)` để **kết quả lặp lại được** (reproducibility) — nguyên tắc rất quan trọng trong nghiên cứu.

In [12]:
rng = np.random.default_rng(seed=42)

print(rng.random(3))                       # 3 số thực đều trong [0, 1)
print(rng.integers(0, 10, size=5))         # 5 số nguyên trong [0, 10)
print(rng.normal(loc=0, scale=1, size=3))  # phân phối chuẩn N(0, 1)

# Cùng seed -> cùng kết quả
print(np.random.default_rng(1).random(2))
print(np.random.default_rng(1).random(2))

# Xáo trộn thứ tự chỉ số rồi chia train/test 80/20
indices = rng.permutation(10)
print(indices)
print("train:", indices[:8], "| test:", indices[8:])

[0.77395605 0.43887844 0.85859792]
[0 6 2 0 5]
[ 0.1278404  -0.31624259 -0.01680116]
[0.51182162 0.9504637 ]
[0.51182162 0.9504637 ]
[7 0 4 2 3 5 1 8 6 9]
train: [7 0 4 2 3 5 1 8] | test: [6 9]


## Bài tập thực hành

1. Tạo array `a` chứa các số từ 1 đến 10 bằng `np.arange`. In ra `shape`, `dtype`, và tổng các phần tử.
2. Từ `a` ở bài 1, tạo array mới chỉ gồm các số **chẵn** bằng boolean mask, rồi in ra.
3. Tạo ma trận `m` shape `(3, 4)` chứa các số từ 0 đến 11 (dùng `arange` + `reshape`). In ra: cột cuối cùng, và tổng của **từng hàng**.
4. Cho `temps = np.array([30.5, 28.0, 35.2, 31.8, 26.4])` (độ C). Chuyển sang độ F bằng công thức `F = C * 9/5 + 32` trong **một dòng**, không dùng vòng `for`.
5. Cho `X = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])`. Dùng broadcasting để **trừ đi giá trị nhỏ nhất của từng cột** (min-centering), in ra kết quả.
6. Cho `probs = np.array([[0.2, 0.5, 0.3], [0.7, 0.2, 0.1], [0.1, 0.1, 0.8]])` và `labels = np.array([1, 0, 1])`. Dùng `argmax` để lấy dự đoán, rồi tính **accuracy** = tỉ lệ dự đoán đúng.
7. **(Nâng cao)** Dùng `rng = np.random.default_rng(0)` tạo ma trận `A` shape `(5, 3)` từ `rng.random`. Không dùng vòng `for`, hãy tính ma trận khoảng cách **Euclid** giữa mọi cặp hàng — kết quả shape `(5, 5)`. Gợi ý: `A[:, None, :] - A[None, :, :]` cho shape `(5, 5, 3)`.

In [13]:
# Bài 1: arange, shape, dtype, sum

In [14]:
# Bài 2: lọc số chẵn bằng boolean mask

In [15]:
# Bài 3: reshape (3, 4), cột cuối, tổng từng hàng

In [16]:
# Bài 4: độ C -> độ F

In [17]:
# Bài 5: trừ min từng cột (broadcasting)

In [18]:
# Bài 6: argmax + accuracy

In [19]:
# Bài 7 (nâng cao): ma trận khoảng cách Euclid

## Gợi ý

- Bài 2: `a[a % 2 == 0]`.
- Bài 3: cột cuối là `m[:, -1]`; tổng từng hàng là `m.sum(axis=1)`.
- Bài 5: `X - X.min(axis=0)` — `X.min(axis=0)` có shape `(3,)`, broadcasting kéo nó thành `(3, 3)`.
- Bài 6: `preds = probs.argmax(axis=1)`; accuracy = `(preds == labels).mean()` (vì `True` = 1, `mean` chính là tỉ lệ).
- Bài 7: `diff = A[:, None, :] - A[None, :, :]` rồi `np.sqrt((diff ** 2).sum(axis=-1))`.

## Lời giải mẫu

⚠️ Chỉ chạy ô dưới đây **sau khi** bạn đã tự làm xong 7 bài tập ở trên.

In [20]:
# --- Lời giải mẫu ---
import numpy as np

# Bài 1
a = np.arange(1, 11)
print(a.shape, a.dtype, a.sum())

# Bài 2
print(a[a % 2 == 0])

# Bài 3
m = np.arange(12).reshape(3, 4)
print(m[:, -1])
print(m.sum(axis=1))

# Bài 4
temps = np.array([30.5, 28.0, 35.2, 31.8, 26.4])
print(temps * 9 / 5 + 32)

# Bài 5
X = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
print(X - X.min(axis=0))

# Bài 6
probs = np.array([[0.2, 0.5, 0.3], [0.7, 0.2, 0.1], [0.1, 0.1, 0.8]])
labels = np.array([1, 0, 1])
preds = probs.argmax(axis=1)
print(preds)
print("accuracy:", (preds == labels).mean())

# Bài 7
rng = np.random.default_rng(0)
A = rng.random((5, 3))
diff = A[:, None, :] - A[None, :, :]       # (5, 5, 3)
dist = np.sqrt((diff ** 2).sum(axis=-1))   # (5, 5)
print(dist.round(2))
print("đường chéo (khoảng cách tới chính nó):", np.diag(dist))

(10,) int64 55
[ 2  4  6  8 10]
[ 3  7 11]
[ 6 22 38]
[86.9  82.4  95.36 89.24 79.52]
[[0. 0. 0.]
 [3. 3. 3.]
 [6. 6. 6.]]
[1 0 2]
accuracy: 0.6666666666666666
[[0.   1.2  0.68 0.62 0.76]
 [1.2  0.   0.7  1.29 1.16]
 [0.68 0.7  0.   0.64 0.76]
 [0.62 1.29 0.64 0.   1.07]
 [0.76 1.16 0.76 1.07 0.  ]]
đường chéo (khoảng cách tới chính nó): [0. 0. 0. 0. 0.]


## Tổng kết

Bạn đã học được:
- ✅ Vì sao NumPy nhanh: lưu số liền kề, cùng kiểu, tính bằng code C — tránh vòng `for`
- ✅ `shape`, `ndim`, `dtype` và các hàm tạo array (`zeros`, `ones`, `arange`, `linspace`, `eye`)
- ✅ Indexing đa chiều, slicing (là **view**, dùng `.copy()` khi cần bản sao), boolean mask
- ✅ Vectorization và hàm tổng hợp theo `axis` (`axis=0`: mỗi cột, `axis=1`: mỗi hàng)
- ✅ Broadcasting: so sánh shape từ cuối về đầu, bằng nhau hoặc bằng 1 là tương thích
- ✅ `reshape`, `.T`, `np.newaxis`, `concatenate`/`stack`, số ngẫu nhiên với `default_rng(seed)`

**Bài tiếp theo:** Bài 2 — Pandas: `Series`, `DataFrame`, chọn cột/hàng với `loc`/`iloc`, lọc và biến đổi dữ liệu dạng bảng. Làm xong bài tập ở trên rồi báo mình nhé!